本次作业以垃圾邮件分类任务为基础，要求提取文本特征并使用朴素贝叶斯算法进行垃圾邮件识别（调用已有工具包或自行实现）。

### 任务介绍
电子邮件是互联网的一项重要服务，在大家的学习、工作和生活中会广泛使用。但是大家的邮箱常常被各种各样的垃圾邮件填充了。有统计显示，每天互联网上产生的垃圾邮件有几百亿近千亿的量级。因此，对电子邮件服务提供商来说，垃圾邮件过滤是一项重要功能。而朴素贝叶斯算法在垃圾邮件识别任务上一直表现非常好，至今仍然有很多系统在使用朴素贝叶斯算法作为基本的垃圾邮件识别算法。

本次实验数据集来自[Trec06](https://plg.uwaterloo.ca/cgi-bin/cgiwrap/gvcormac/foo06)的中文垃圾邮件数据集，目录解压后包含三个文件夹，其中data目录下是所有的邮件（未分词），已分词好的邮件在data_cut目录下。邮件分为邮件头部分和正文部分，两部分之间一般有空行隔开。标签数据在label文件夹下，文件中每行是标签和对应的邮件路径。‘spam’表示垃圾邮件，‘ham’表示正常邮件。

本次实验

基本要求：
1. 提取正文部分的文本特征；
2. 划分训练集和测试集（可以借助工具包。一般笔记本就足够运行所有数据，认为实现困难或算力不够的同学可以采样一部分数据进行实验。）；
3. 使用朴素贝叶斯算法完成垃圾邮件的分类与预测，要求测试集准确率Accuracy、精准率Precision、召回率Recall均高于0.9（本次实验可以使用已有的一些工具包完成如sklearn）；
4. 对比特征数目（词表大小）对模型效果的影响；
5. 提交代码和实验报告。

扩展要求：
1. 邮件头信息有时也可以协助判断垃圾邮件，欢迎学有余力的同学们尝试；
2. 尝试自行实现朴素贝叶斯算法细节；
3. 尝试对比不同的概率计算方法。

### 导入工具包

In [ ]:
%matplotlib inline
import os  # 导入操作系统相关功能的模块，用于文件和目录操作
import re  # 导入正则表达式模块，用于字符串的模式匹配和替换
import glob  # 导入用于查找符合特定规则的文件路径名的模块
import time  # 导入时间模块，用于获取当前时间和进行时间相关的操作
import numpy as np  # 导入NumPy库，用于高效的数值计算和数组操作
import pandas as pd  # 导入Pandas库，用于数据处理和分析
import matplotlib.pyplot as plt  # 导入Matplotlib的pyplot模块，用于绘制各种图表
import seaborn as sns  # 导入Seaborn库，用于绘制更美观的统计图表
import logging  # 导入日志记录模块，用于记录程序运行过程中的信息
import json  # 导入JSON模块，用于处理JSON数据
from typing import Union  # 导入Union类型用于联合类型注解
from joblib import Parallel, delayed  # 从Joblib库中导入并行计算和延迟执行的功能
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
)  # 从sklearn库中导入文本特征提取的向量化器
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
)  # 从sklearn库中导入数据集划分\随机网格搜索功能模块
from sklearn.naive_bayes import (
    MultinomialNB,
    ComplementNB,
    BernoulliNB,
)  # 从sklearn库中导入朴素贝叶斯分类起的不同变体
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)  # 从sklearn库中导入各种评估指标，用于评估模型性能
from sklearn.preprocessing import (
    LabelEncoder,
)  # 从sklearn库中导入标签编码器，用于将分类标签转化为数值
from collections import Counter  # 导入用于统计元素出现次数的Counter类
from scipy.sparse import hstack  # 从scipy.sparse导入hstack库用于合并特征矩阵
from scipy.stats import (
    loguniform,
    uniform,
)  # 从scipy.stats中导入loguniform模块和uniform模块，用于定义对数均匀分布的超参数搜索空间和均匀分布的超参数搜索空间

### 环境配置

In [ ]:
# 配置Matplotlib使用中文字体，确保中文可以正常显示
plt.rcParams["font.family"] = ["SimHei"]

# 配置Matplotlib正确显示负号
plt.rcParams["axes.unicode_minus"] = False

# 设置Matplotlib的字体大小
plt.rcParams["font.size"] = 12

# 设置Seaborn的绘图风格为白色网格风格
sns.set_theme(style="whitegrid", font="SimHei")  # 显式指定字体

# 配置日志记录的基本信息，设置日志级别为INFO，指定日志格式
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

# 创建一个名为'SpamClassifier'的日志记录器
logger = logging.getLogger("SpamClassifier")

# 配置并行处理使用所有可用的CPU线程数
N_JOBS = 1  # 使用单线程运行

# 设置随机数种子
RANDOM_SEED = 2025

### 定义工具函数

In [ ]:
def timing_decorator(func) -> callable:  # type: ignore
    """
    计算函数执行时间的装饰器

    :param func: 被装饰的函数
    :return: 包装后的函数
    """

    def wrapper(*args, **kwargs) -> any:  # type: ignore
        start_time = time.time()  # 记录函数开始执行的时间
        result = func(*args, **kwargs)  # 调用被装饰的函数
        end_time = time.time()  # 记录函数执行结束的时间
        logger.info(
            f"{func.__name__} 执行时间: {end_time - start_time:.2f} 秒"
        )  # 记录函数执行时间
        return result  # 返回被装饰函数的执行结果

    return wrapper  # 返回包装后的函数

### 定义读取邮件的类

In [ ]:
class EmailDataset:
    """
    邮件数据集的加载和预处理
    """

    def __init__(self, base_path="trec06c-utf8") -> None:
        """
        初始化EmailDataset类

        :param base_path: 数据集的基础路径，默认为'trec06c-utf8'
        """

        self.base_path = base_path  # 保存数据集的基础路径
        self.label_path = os.path.join(
            base_path, "label", "index"
        )  # 拼接标签文件的路径
        self.data_path = os.path.join(base_path, "data_cut")  # 拼接数据文件的路径
        self.emails = []  # 用于存储邮件内容
        self.labels = []  # 用于存储邮件标签
        self.header_features = []  # 用于存储邮件头特征

    def _resolve_path(self, original_path) -> str:
        """
        解决路径不一致问题

        :param original_path: 原始文件路径
        :return: 解析后的问价路径，如果文件不存在则返回None
        """

        rel_path = original_path.replace("../data/", "")
        pattern = os.path.join(self.base_path, "data_cut", rel_path) + "*"
        matched = glob.glob(pattern)
        if matched and os.path.exists(matched[0]):
            return matched[0]
        else:
            logger.warning(f"文件未找到: {original_path}")
            return None  # type: ignore

    def _extract_header_features(self, content: str) -> dict:
        """
        提取邮件头特征

        :param content: 邮件内容
        :return: 包含邮件头特征的字典
        """

        features = {}  # 初始化特征字典

        # 提取发件人域名
        from_match = re.search(r"From:\s*.*@([a-zA-Z0-9.-]+)", content, re.IGNORECASE)
        features["发件人域名"] = from_match.group(1).lower() if from_match else "未知"

        # 检测是否包含特定可疑头
        features["有高优先级标记"] = (
            1 if re.search(r"X-Priority:\s*1\b", content, re.IGNORECASE) else 0
        )
        features["有MIME编码"] = (
            1
            if re.search(
                r"Content-Transfer-Encoding:\s*base64\b", content, re.IGNORECASE
            )
            else 0
        )

        # 计算接收服务器的数量
        received_count = len(re.findall(r"Received:", content, re.IGNORECASE))
        features["接收服务器的数量"] = received_count

        # 检测是否包含可疑关键词
        features["有可疑关键词"] = (
            1
            if re.search(
                r"(免费|发票|促销|优惠|代开|增值税|地税|国税|真票|税务局)",
                content[:1500],
            )
            else 0
        )

        return features

    def _extract_content(self, file_path) -> tuple[str, dict]:
        """
        提取邮件内容和头部特征

        :param file_path: 文件路径
        :return: 邮件正文内容和头部特征字典
        """

        try:
            with open(
                file_path, "r", encoding="utf-8", errors="ignore"
            ) as f:  # 以只读模式打开文件
                content = f.read()  # 读取文件内容
                header_features = self._extract_header_features(
                    content
                )  # 提取邮件头特征
                match = re.search(r"\n\s*\n", content)  # 定位空行
                body = (
                    content[match.end() :].strip() if match else content
                )  # 提取邮件正文
                return body, header_features

        except Exception as e:
            logger.warning(
                f"读取 {file_path} 时出错: {str(e)}"
            )  # 记录读取文件时的错误信息
            return "", {}  # 返回空正文和空头部特征

    @timing_decorator
    def load_data(self, sample_size: int = None) -> tuple[list, list, list]:  # type: ignore
        """
        加载标签和邮件内容

        :param sample_size: 抽样的样本数量，如果为None则是用全部数据
        :return: 邮件内容、标签和头部组成的元组
        """

        logger.info(f"加载标签文件: {self.label_path}")  # 记录加载标签文件的信息

        # 读取标签文件
        label_data = []
        with open(self.label_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = (
                    line.strip().split()
                )  # 对每一行标签以空格为分隔符分割出标签和路径
                if len(parts) >= 2:
                    label, path = parts[0], parts[1]
                    resolved_path = self._resolve_path(path)
                    if resolved_path:
                        label_data.append((label, resolved_path))

        # 抽样
        if sample_size and sample_size < len(label_data):
            np.random.seed(RANDOM_SEED)
            indices = np.random.choice(len(label_data), sample_size, replace=False)
            label_data = [label_data[i] for i in indices]

        logger.info(
            f"找到 {len(label_data)} 封带标签的邮件"
        )  # 记录找到带标签的邮件数量

        # 加载邮件内容
        results = Parallel(n_jobs=N_JOBS)(
            delayed(self._extract_content)(path) for _, path in label_data
        )

        # 解包结果
        self.emails = [res[0] for res in results]
        self.header_features = [res[1] for res in results]
        self.labels = [1 if label == "spam" else 0 for label, _ in label_data]

        logger.info(f"成功加载 {len(self.emails)} 封邮件")  # 记录成功加载的邮件数量
        return self.emails, self.labels, self.header_features

    def get_class_distribution(self) -> dict:
        """
        获取类别分布

        :return: 包含垃圾邮件、正常邮件和总计数量的字典
        """

        counter = Counter(self.labels)  # 统计标签的分布
        return {
            "垃圾邮件": counter[1],
            "正常邮件": counter[0],
            "总计": len(self.labels),
        }

    def __repr__(self) -> str:
        """
        返回数据集的字符串表示

        :return: 包含数据集信息的字符串
        """

        dist = self.get_class_distribution()
        return f"邮件数据集(垃圾邮件={dist['垃圾邮件']}, 正常邮件={dist['正常邮件']}, 总计={dist['总计']})"

### 定义特征工程的类

In [ ]:
class FeatureExtractor:
    """
    特征提取器
    """

    def __init__(
        self,
        vectorizer_type: str = "tfidf",
        max_features: int = 10000,
        ngram_range: tuple = (1, 1),
        binary: bool = False,
    ) -> None:
        """
        初始化FeatureExtractor类

        :param vectorizer_type: 向量化器的类型，默认为'tfidf'
        :param max_features: 最大特征数量，默认为10000
        :param ngram_range: n-gram的范围，默认为(1, 1)
        """

        self.vectorizer_type = vectorizer_type  # 向量化器的类型
        self.max_features = max_features  # 最大特征数
        self.ngram_range = ngram_range  # n-gram的范围
        self.binary = binary  # 支持伯努利分类器的二值特征
        self.vectorizer = None  # 向量化器
        self.header_encoder = None  # 邮件头特征编码器
        self.feature_names = []  # 特征名称

    def create_vectorizer(self) -> CountVectorizer | TfidfVectorizer:
        """
        创建向化器

        :return: 向量化器对象
        """

        if self.vectorizer_type == "tfidf":
            return TfidfVectorizer(
                max_features=self.max_features, ngram_range=self.ngram_range
            )
        else:  # 'tfidf'为默认
            return CountVectorizer(
                max_features=self.max_features,
                ngram_range=self.ngram_range,
                binary=self.binary,
            )

    def extract_text_features(self, emails: list) -> any:
        """
        提取文本特征

        :param emails: 邮件内容列表
        :return: 文本特征矩阵
        """

        # 如果是首次调用，需要先拟合向量化器
        if self.vectorizer is None:
            self.vectorizer = self.create_vectorizer()
            text_features = self.vectorizer.fit_transform(
                emails
            )  # 首次调用使用fit_transform
            self.feature_names = self.vectorizer.get_feature_names_out()
        else:
            text_features = self.vectorizer.transform(emails)  # 后续调用使用transform

        logger.debug(f"文本特征矩阵形状: {text_features.shape}")

        return text_features

    def extract_header_features(self, header_features: list) -> np.ndarray:
        """
        提取邮件头特征

        :param header_features: 邮件头特征列表
        :return: 邮件头特征矩阵
        """

        header_df = pd.DataFrame(header_features)  # 转换为DataFrame以便处理

        # 对分类特征进行编码
        if self.header_encoder is None:
            self.header_encoder = {}
            for col in header_df.columns:
                if header_df[col].dtype == "object":  # 分类器特征
                    le = LabelEncoder()
                    header_df[col] = le.fit_transform(header_df[col].astype(str))
                    self.header_encoder[col] = le
                else:
                    header_df[col] = header_df[col].astype(float)
        else:
            for col in header_df.columns:
                if col in self.header_encoder:  # 分类特征
                    header_df[col] = self.header_encoder[col].transform(
                        header_df[col].astype(str)
                    )
                else:
                    header_df[col] = header_df[col].astype(float)

        return header_df.values

    def extract_all_features(self, emails: list, header_features: list) -> any:
        """
        提取文本和邮件头特征

        :param emails: 邮件内容列表
        :param header_features: 邮件头特征列表
        :return: 合并后的特征矩阵
        """

        text_features = self.extract_text_features(emails)
        header_features = self.extract_header_features(header_features)

        return hstack([text_features, header_features])  # 合并特征

### 定义垃圾邮件分类器的类

In [ ]:
class SpamClassifier:
    """
    垃圾邮件分类器
    """

    def __init__(
        self,
        model_type: str = "multinomial",
        feature_extractor: FeatureExtractor = None,
    ) -> None:
        """
        初始化SpamClassifier类

        :param model_type: 模型类型，默认为'multinomial'
        :param feature_extractor: 特征提取器对象，默认为None
        """

        self.model_type = model_type  # 模型类型
        self.feature_extractor = feature_extractor or FeatureExtractor()  # 特征提取器
        self.model = self._create_model()  # 创建模型
        self.trained = False  # 模型是否训练的标志
        self.best_params_ = None  # 存储最佳参数

    def _create_model(self) -> MultinomialNB | ComplementNB | BernoulliNB:
        """
        创建模型

        :return: 模型对象
        """

        if self.model_type == "complement":
            return ComplementNB()
        elif self.model_type == "bernoulli":
            return BernoulliNB(binarize=0.5)
        else:
            return MultinomialNB()  # multinomial为默认

    @timing_decorator
    def train(self, X_train: Union[list, any], y_train: list) -> any:
        """
        训练模型

        :param X_train: 训练数据，可以是原始文本或特征矩阵
        :param y_train: 训练标签
        :return: 训练数据的特征矩阵
        """

        # 检查训练数据标签分布
        label_counter = Counter(y_train)
        logger.info(f"训练数据标签分布: {label_counter}")

        # 如果X_train是原始文本，需要先提取特征
        if isinstance(X_train[0], str):
            X_train = self.feature_extractor.extract_text_features(X_train)

        logger.info("训练模型中...")  # 记录训练开始信息
        self.model.fit(X_train, y_train)  # 训练模型
        self.trained = True  # 标记模型已训练
        logger.info("模型训练完成")  # 记录训练完成信息

        # 获取最重要的特征
        if hasattr(self.model, "feature_log_prob_"):
            self.feature_importance = (
                self.model.feature_log_prob_[1] - self.model.feature_log_prob_[0]
            )
        else:
            self.feature_importance = None

        return X_train

    @timing_decorator
    def hyperparameter_tuning(
        self,
        X_train: any,
        y_train: list,
        param_distributions: dict,
        n_iter: int = 500,
        cv: int = 5,
        random_state=RANDOM_SEED,
    ) -> float:
        """
        使用随机网格搜索进行超参数优化

        :param X_train: 训练数据的特征矩阵
        :param y_train: 训练标签
        :param param_distributions: 超参数分布字典
        :param n_iter: 随机搜索的迭代次数，默认为500
        :param cv: 交叉验证的折数，默认为5
        :param random_state: 随机种子
        :return: 最佳F1分数
        """

        # 创建搜索对象
        random_search = RandomizedSearchCV(
            estimator=self.model,
            param_distributions=param_distributions,
            n_iter=n_iter,
            cv=cv,
            scoring="f1_weighted",  # 备选'roc_auc', 'recall'
            n_jobs=N_JOBS,
            random_state=random_state,
            verbose=1,
        )

        # 执行搜索
        random_search.fit(X_train, y_train)

        # 更新模型和参数
        self.model = random_search.best_estimator_
        self.best_params_ = random_search.best_params_
        self.trained = True

        logger.info(
            f"超参数优化完成～，交叉验证最佳参数组合为: {self.best_params_}"
        )  # 记录超参数优化完成信息
        logger.info(
            f"最佳参数组合在交叉验证中的平均F1加权分数为: {random_search.best_score_:.4f}"
        )  # 记录最佳F1分数

        return random_search.best_score_

    def predict(self, X: Union[list, any]) -> np.ndarray:
        """
        预测邮件分类

        :param X: 待预测的数据，可以是原始文本或特征矩阵
        :return: 预测结果数组
        """

        if not self.trained:
            raise RuntimeError("模型尚未训练")  # 若模型未训练，抛出运行时错误

        if isinstance(X[0], str):  # 如果是原始文本，需要先提取特征
            X = self.feature_extractor.extract_text_features(X)

        return self.model.predict(X)

    def predict_proba(self, X: Union[list, any]) -> np.ndarray:
        """
        预测概率

        :param X: 待预测的数据，可以是原始文本或特征矩阵
        :return: 预测为垃圾邮件的概率数组
        """

        if not self.trained:
            raise RuntimeError("模型尚未训练")  # 若模型未训练，抛出运行时错误

        if isinstance(X[0], str):  # 如果X是原始文本，需要先提取特征
            X = self.feature_extractor.extract_text_features(X)

        return self.model.predict_proba(X)[:, 1]  # 返回垃圾邮件的概率

    def _calculate_auc(self, y_true: list, y_proba: np.ndarray) -> float:
        """
        计算AUC

        :param y_true: 真实标签
        :param y_praba: 预测概率
        :return: AUC值
        """

        fpr, tpr, _ = roc_curve(y_true, y_proba)
        return auc(fpr, tpr)

    def evaluate(self, X_test: Union[list, any], y_test: list) -> dict:
        """
        评估模型性能

        :param X_text: 测试数据，可以是原始文本或特征矩阵
        :param y_test: 测试标签
        :return: 包含评估指标的字典
        """

        if not self.trained:
            raise RuntimeError("模型尚未训练")  # 若模型未训练，抛出运行时错误

        y_pred = self.predict(X_test)
        y_proba = self.predict_proba(X_test)

        metrics = {  # 计算各项指标
            "混淆矩阵": confusion_matrix(y_test, y_pred),
            "准确率": accuracy_score(y_test, y_pred),
            "查准率": precision_score(y_test, y_pred),
            "查全率": recall_score(y_test, y_pred),
            "F1分数": f1_score(y_test, y_pred),
            "AUC": self._calculate_auc(y_test, y_proba),
        }

        return metrics

    def get_feature_importance_df(self, top_n: int = 16) -> pd.DataFrame:
        """
        获取特征重要性的DataFrame

        :param top_n: 现实前n个特征，默认为16
        :return: 包含特征重要性的DataFrame
        """

        if not self.trained or self.feature_importance is None:
            raise RuntimeError("特征重要性不可用")  # 若特征重要性不可用，抛出运行时错误

        feature_names = self.feature_extractor.feature_names  # 获取特征名称

        # 获取最重要的特征
        top_indices = np.argsort(np.abs(self.feature_importance))[-top_n:][::-1]
        top_features = [
            (feature_names[i], self.feature_importance[i])
            for i in top_indices
            if i < len(feature_names)
        ]

        return pd.DataFrame(top_features, columns=["特征", "重要性"])

    def display_feature_importance(self, top_n: int = 16) -> None:
        """
        显示特征重要性

        :param top_n: 显示前n个重要的特征，默认为16
        """

        try:
            if not hasattr(self.model, "feature_log_prob_"):
                logger.warning("当前模型不支持特征重要性计算")
                return

            df = self.get_feature_importance_df(top_n)

            plt.figure(figsize=(12, 8))
            sns.barplot(
                x="重要性",
                y="特征",
                hue="特征",
                palette="viridis",
                legend=False,
                data=df.sort_values(by="重要性", ascending=False),
            )
            plt.title(f"邮件正文中最重要的 {top_n} 个特征", color="purple")
            plt.tight_layout()
            plt.show()

        except RuntimeError as e:
            logger.waring(str(e))  # 记录运行时错误信息

### 定义实验运行的类

In [ ]:
class ExperimentRunner:
    """
    执行实验并分析结果
    """

    def __init__(self, dataset: EmailDataset) -> None:
        """
        初始化ExperimentRunner类

        :param dataset: 邮件数据集对象
        """

        self.dataset = dataset  # 邮件数据集
        self.results = {}  # 存储实验结果
        self.visualizations = {}  # 存储可视化结果

    @timing_decorator
    def run_baseline_experiment(
        self, test_size: float = 0.2, random_state: int = RANDOM_SEED
    ) -> dict:
        """
        运行基线实验

        :param test_size: 测试集的比例，默认为0.2
        :param random_state: 随机种子
        :return: 包含评估指标的字典
        """

        logger.info("运行基线实验")  # 记录基线实验开始信息

        X_train, X_test, y_train, y_test = train_test_split(  # 划分数据集
            self.dataset.emails,
            self.dataset.labels,
            test_size=test_size,
            random_state=random_state,
            stratify=self.dataset.labels,
        )

        classifier = SpamClassifier(model_type="multinomial")  # 训练模型
        classifier.train(X_train, y_train)

        metrics = classifier.evaluate(X_test, y_test)  # 评估模型

        self.results["baseline"] = {  # 存储结果
            "metrics": metrics,
            "classifier": classifier,
            "top_features": classifier.get_feature_importance_df(16),
        }

        self._generate_baseline_visualizations(
            metrics, y_test, classifier.predict_proba(X_test)
        )  # 生成可视化

        return metrics

    @timing_decorator
    def run_feature_size_experiment(
        self,
        feature_sizes: list,
        test_size: float = 0.2,
        random_state: int = RANDOM_SEED,
    ) -> pd.DataFrame:
        """
        对比不同特征数量的影响

        :param feature_sizes: 特征数量列表
        :param test_size: 测试集的比例
        :param random_state: 随机种子
        :return: 包含实验结果的DataFrame
        """

        logger.info("运行特征数量影响实验")  # 记录特征数量影响实验开始信息

        X_train, X_test, y_train, y_test = train_test_split(  # 划分数据集
            self.dataset.emails,
            self.dataset.labels,
            test_size=test_size,
            random_state=random_state,
            stratify=self.dataset.labels,
        )

        results = []

        for size in feature_sizes:
            logger.info(f"测试特征数量: {size}")  # 记录当前测试的特征数量

            feature_extractor = FeatureExtractor(
                max_features=size
            )  # 创建新的特征提取器

            classifier = SpamClassifier(
                model_type="multinomial", feature_extractor=feature_extractor
            )  # 训练模型
            classifier.train(X_train, y_train)

            metrics = classifier.evaluate(X_test, y_test)  # 评估模型

            results.append(
                {
                    "设置特征数": size,
                    "准确率": metrics["准确率"],
                    "查准率": metrics["查准率"],
                    "查全率": metrics["查全率"],
                    "AUC": metrics["AUC"],
                }
            )

        self.results["feature_size"] = pd.DataFrame(results)

        self._generate_feature_size_visualizations(
            self.results["feature_size"]
        )  # 生成可视化

        return self.results["feature_size"]

    @timing_decorator
    def run_probability_method_experiment(
        self, test_size: float = 0.2, random_state: int = RANDOM_SEED
    ) -> pd.DataFrame:
        """
        对比不同的概率计算方法

        :param test_size: 测试集的比例，默认为0.2
        :param random_state: 随机种子
        :return: 包含实验结果的DataFrame
        """

        logger.info("运行概率计算方法对比实验")  # 记录概率计算方法对比实验开始信息

        # 划分数据集
        X_train, X_test, y_train, y_test = train_test_split(
            self.dataset.emails,
            self.dataset.labels,
            test_size=test_size,
            random_state=random_state,
            stratify=self.dataset.labels,
        )

        results = []
        classifiers = []

        methods = [
            ("多项式朴素贝叶斯", "multinomial"),
            ("补充朴素贝叶斯", "complement"),
            ("伯努利朴素贝叶斯", "bernoulli"),
        ]

        for name, model_type in methods:
            logger.info(f"测试方法: {name}")  # 记录当前测试的方法

            if model_type == "bernoulli":  # 为伯努利分类器使用不同的特征提取方式
                feature_extractor = FeatureExtractor(
                    vectorizer_type="count", binary=True
                )
                classifier = SpamClassifier(
                    model_type=model_type, feature_extractor=feature_extractor
                )
            else:
                classifier = SpamClassifier(model_type=model_type)

            classifier.train(X_train, y_train)
            metrics = classifier.evaluate(X_test, y_test)

            results.append(
                {
                    "方法": name,
                    "准确率": metrics["准确率"],
                    "查准率": metrics["查准率"],
                    "查全率": metrics["查全率"],
                    "F1分数": metrics["F1分数"],
                    "AUC": metrics["AUC"],
                }
            )

            classifiers.append(classifier)

        self.results["probability_methods"] = pd.DataFrame(results)
        self.results["classifiers"] = classifiers

        self._generate_probability_method_visualizations(
            self.results["probability_methods"]
        )  # 生成可视化

        return self.results["probability_methods"]

    @timing_decorator
    def run_hyperparameter_tuning_experiment(
        self, test_size: float = 0.2, random_state: int = RANDOM_SEED
    ) -> pd.DataFrame:
        """
        运行超参数调优实验

        :param test_size: 测试集的比例，默认为0.2
        :param random_state: 随机种子
        :return: 包含实验结果的DataFrame
        """

        logger.info("运行超参数调优实验")  # 记录超参数调优实验开始信息

        # 划分数据集
        X_train, X_test, y_train, y_test = train_test_split(
            self.dataset.emails,
            self.dataset.labels,
            test_size=test_size,
            random_state=random_state,
            stratify=self.dataset.labels,
        )

        # 定义不同模型的参数分布
        param_distributions = {
            "multinomial": {"alpha": loguniform(1e-10, 1), "fit_prior": [True, False]},
            "complement": {
                "alpha": loguniform(1e-10, 1),
                "norm": [True, False],
                "fit_prior": [True, False],
            },
            "bernoulli": {
                "alpha": loguniform(1e-10, 1),
                "binarize": uniform(0.0, 1.0),
                "fit_prior": [True, False],
            },
        }

        results = []
        best_classifiers = {}

        for model_type in ["multinomial", "complement", "bernoulli"]:
            logger.info(
                f"\n=== 优化 {model_type} 贝叶斯分类器 ==="
            )  # 记录当前优化的模型类型

            if model_type == "bernoulli":  # 创建分类器和特征提取器
                feature_extractor = FeatureExtractor(
                    vectorizer_type="count",
                    max_features=10000,
                    ngram_range=(1, 2),
                    binary=True,
                )
            else:
                feature_extractor = FeatureExtractor(
                    vectorizer_type="tfidf", max_features=10000, ngram_range=(1, 2)
                )

            classifier = SpamClassifier(
                model_type=model_type, feature_extractor=feature_extractor
            )

            X_train_features = feature_extractor.extract_text_features(
                X_train
            )  # 提取训练特征

            best_score = classifier.hyperparameter_tuning(  # 进行超参数优化
                X_train_features,
                y_train,
                param_distributions=param_distributions[model_type],
                random_state=random_state,
            )

            X_test_features = feature_extractor.extract_text_features(
                X_test
            )  # 评估测试集性能
            metrics = classifier.evaluate(X_test_features, y_test)

            results.append(
                {
                    "方法": model_type,
                    "最佳参数": json.dumps(classifier.best_params_),
                    "最佳F1": best_score,
                    "测试准确率": metrics["准确率"],
                    "测试查准率": metrics["查准率"],
                    "测试查全率": metrics["查全率"],
                    "测试F1分数": metrics["F1分数"],
                    "测试AUC": metrics["AUC"],
                }
            )

            best_classifiers[model_type] = classifier

        self.results["hyperparameter_tuning"] = pd.DataFrame(results)  # 存储结果
        self.results["tuned_classifiers"] = best_classifiers

        self._generate_hyperparam_visualizations(
            self.results["hyperparameter_tuning"]
        )  # 生成可视化

        return self.results["hyperparameter_tuning"]

    def _generate_baseline_visualizations(
        self, metrics: dict, y_true: list, y_proba: np.ndarray
    ) -> None:
        """
        生成基线实验的可视化

        :param metrics: 包含评估指标的字典
        :param y_true: 真实标签
        :param y_proba: 预测概率
        """

        plt.figure(figsize=(10, 8))  # 混淆矩阵图
        sns.heatmap(
            metrics["混淆矩阵"],
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["预测正常", "预测垃圾"],
            yticklabels=["实际正常", "实际垃圾"],
        )
        plt.title("混淆矩阵", color="purple")
        plt.xlabel("预测类别", color="blue")
        plt.ylabel("真实类别", color="green")
        plt.tight_layout()
        plt.gca().tick_params(colors="black")
        plt.show()

        fpr, tpr, _ = roc_curve(y_true, y_proba)  # ROC曲线图
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(8, 8))
        plt.plot(
            fpr, tpr, color="darkorange", lw=2, label=f"ROC曲线 (AUC = {roc_auc:.2f})"
        )
        plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
        plt.xlim([0.0, 1.01])
        plt.ylim([0.0, 1.01])
        plt.xlabel("假正例率", color="blue")
        plt.ylabel("真正例率", color="green")
        plt.title("ROC曲线", color="purple")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.gca().tick_params(colors="black")
        plt.show()

        precision, recall, _ = precision_recall_curve(
            y_true, y_proba
        )  # 查准率-查全率曲线
        plt.figure(figsize=(8, 8))
        plt.plot(recall, precision, color="red", lw=2)
        plt.xlabel("查全率", color="blue")
        plt.ylabel("查准率", color="green")
        plt.title("查准率-查全率曲线", color="purple")
        plt.xlim([0.0, 1.01])
        plt.ylim([0.0, 1.01])
        plt.tight_layout()
        plt.gca().tick_params(colors="black")
        plt.show()

        self.results["baseline"]["classifier"].display_feature_importance(16)

    def _generate_feature_size_visualizations(self, df: pd.DataFrame) -> None:
        """
        生成特征数量实验的可视化

        :param df: 包含实验结果的DataFrame
        """

        plt.figure(figsize=(12, 8))
        feature_sizes = df["设置特征数"].tolist()

        x_values = np.array(feature_sizes)  # 直接使用特征数量值作为x轴

        plt.subplot(2, 1, 1)  # 综合对比
        plt.plot(x_values, df["准确率"], "o-", label="准确率")
        plt.plot(x_values, df["查准率"], "s-", label="查准率")
        plt.plot(x_values, df["查全率"], "d-", label="查全率")
        plt.xlabel("特征数量", color="blue")
        plt.ylabel("得分", color="green")
        plt.title("特征数量对多项式朴素贝叶斯分类器性能的影响", color="purple")
        plt.xscale("log")

        plt.xticks(x_values, [str(x) for x in feature_sizes], rotation=60)
        plt.grid(True)
        plt.legend()

        plt.subplot(2, 1, 2)  # AUC变化
        plt.plot(x_values, df["AUC"], "s-", color="purple")
        plt.xlabel("特征数量", color="blue")
        plt.ylabel("AUC", color="green")
        plt.title("特征数量对AUC的影响", color="purple")
        plt.xscale("log")

        plt.xticks(x_values, [str(x) for x in feature_sizes], rotation=60)
        plt.grid(True)

        plt.tight_layout()
        plt.show()

    def _generate_probability_method_visualizations(self, df: pd.DataFrame) -> None:
        """
        生成概率方法实验的可视化

        :param df: 包含实验结果的DataFrame
        """

        plt.figure(figsize=(12, 8))

        ax = df.plot(
            x="方法",
            y=["准确率", "查准率", "查全率", "AUC"],
            kind="bar",
            figsize=(14, 8),
            colormap="viridis",
        )
        plt.title("不同朴素贝叶斯方法的性能对比", color="purple")
        plt.xlabel("方法", color="blue")
        plt.ylabel("得分", color="green")
        plt.ylim(0.8, 1.01)
        plt.xticks(rotation=0)
        plt.grid(axis="y", linestyle="--", alpha=0.7)

        for p in ax.patches:
            ax.annotate(
                f"{p.get_height():.3f}",
                (p.get_x() + p.get_width() / 2.0, p.get_height()),
                ha="center",
                va="center",
                xytext=(0, 5),
                textcoords="offset points",
            )

        plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
        plt.tight_layout()
        plt.show()

    def _generate_hyperparam_visualizations(self, df: pd.DataFrame) -> None:
        """
        生成超参数调优结果的可视化

        :param df: 包含实验结果的DataFrame
        """

        plt.figure(figsize=(12, 8))
        metrics = ["测试准确率", "测试查准率", "测试查全率", "测试F1分数", "测试AUC"]
        df_long = pd.melt(
            df, id_vars=["方法"], value_vars=metrics, var_name="指标", value_name="分数"
        )

        plt.figure(figsize=(12, 8))
        ax = sns.barplot(
            x="方法", y="分数", hue="指标", data=df_long, palette="viridis"
        )
        plt.xlabel("方法", color="blue")
        plt.ylabel("分数", color="green")
        plt.title("超参数优化后各模型性能对比", color="purple")
        plt.legend(
            title="指标", loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0.0
        )
        plt.ylim(0.8, 1.01)
        plt.grid(axis="y", linestyle="--", alpha=0.7)

        for p in ax.patches:
            ax.annotate(
                f"{p.get_height():.3f}",
                (p.get_x() + p.get_width() / 2.0, p.get_height()),
                ha="center",
                va="center",
                xytext=(0, 5),
                textcoords="offset points",
                fontsize=9,
            )

        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 8))

        bar_width = 0.35
        x = np.arange(len(df["方法"]))

        rects1 = plt.bar(
            x - bar_width / 2, df["最佳F1"], bar_width, color="skyblue", label="最佳F1"
        )
        rects2 = plt.bar(
            x + bar_width / 2,
            df["测试F1分数"],
            bar_width,
            color="salmon",
            alpha=0.7,
            label="测试F1",
        )

        plt.xlabel("方法", color="blue")
        plt.ylabel("F1分数", color="green")
        plt.title("交叉验证与测试集F1分数对比", color="purple")
        plt.xticks(x, df["方法"])
        plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.ylim(0.8, 1.0)

        def autolabel(rects):
            for rect in rects:
                height = rect.get_height()
                plt.annotate(
                    f"{height:.3f}",
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                )

        autolabel(rects1)
        autolabel(rects2)
        plt.tight_layout()
        plt.show()

    def display_results(self) -> None:
        """
        在Notebook中显示所有实验结果
        """

        dist = self.dataset.get_class_distribution()
        print(f"数据集信息:")
        print(f"   总邮件数: {dist['总计']}")
        print(
            f"   垃圾邮件数: {dist['垃圾邮件']} ({dist['垃圾邮件'] / dist['总计']:.2%})"
        )
        print(
            f"   正常邮件数: {dist['正常邮件']} ({dist['正常邮件'] / dist['总计']:.2%})\n"
        )

        if "baseline" in self.results:
            metrics = self.results["baseline"]["metrics"]
            print("基线实验结果:")
            print(f"  准确率: {metrics['准确率']:.4f}")
            print(f"  查准率: {metrics['查准率']:.4f}")
            print(f"  查全率: {metrics['查全率']:.4f}")
            print(f"  F1分数: {metrics['F1分数']:.4f}")
            print(f"  AUC: {metrics['AUC']:.4f}\n")

            print("最重要的16个特征:")
            display(self.results["baseline"]["top_features"])

        if "feature_size" in self.results:
            df = self.results["feature_size"]
            print("\n特征数量影响实验结果:")
            display(df)

        if "probability_methods" in self.results:
            df = self.results["probability_methods"]
            print("\n概率计算方法对比结果:")
            display(df)

        if "hyperparameter_tuning" in self.results:
            df = self.results["hyperparameter_tuning"]
            print("\n超参数调优实验结果:")
            display(df)

### 运行与结果展示(包含所有基本要求)

In [ ]:
# 初始化数据集
dataset = EmailDataset(base_path="./trec06c-utf8")
emails, labels, headers = dataset.load_data()

In [ ]:
# 初始化实验运行器
experiment = ExperimentRunner(dataset)

In [ ]:
# 实验一：基线模型(多项式朴素贝叶斯分类器)
baseline_metrics = experiment.run_baseline_experiment()

&emsp;&emsp;可以观察到，基线模型已经能够很好的完成垃圾邮件分类任务。基线模型在测试集上的准确率约为<font color = "red">**96.73%**</font>，查准率为<font color = "red">**98.07%**</font>，查全率为<font color = "red">**97.00%**</font>。

In [ ]:
# 实验二：特征数量影响
feature_sizes = [100, 1000, 10000, 100000, 200000, 300000, 400000, 500000]
feature_results_df = experiment.run_feature_size_experiment(feature_sizes)

&emsp;&emsp;可以观察到，在特征数量小于十万的时候，随着特征数量的增加，模型在测试集上的准确率、查全率和AUC也在逐渐地增加，查准率在特征数量为一万处达到高点后略微下降，查准率的峰值可能出现在特征数量一万到十万之间。在特征数量大于十万后，随着特征数量的增加，模型在测试集上的AUC略微下降在特征数量大于二十后趋于稳定，查全率升高在特征数量大于二十万后趋于稳定，准确率略微升高在特征数量大于二十万后趋于稳定，查准率下降在特征数量大于二十万后趋于稳定。

In [ ]:
# 实验三：不同概率计算方法
prob_results_df = experiment.run_probability_method_experiment()

&emsp;&emsp;综合来看，在默认参数下<font color = "red">**多项式朴素贝叶斯**</font>的表现是三种方法中最好的。

In [ ]:
# 实验四：超参数调优
hyperparam_results_df = experiment.run_hyperparameter_tuning_experiment()

&emsp;&emsp;经过超参数优化后，<font color = "purple">**多项式朴素贝叶斯**</font>的综合表现仍然最优。

In [ ]:
# 显示所有实验结果
experiment.display_results()